|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>CUDA graphs<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: capture a step, then survive a changing batch<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cudalib

Capture a decode step as a CUDA graph, then make it survive a batch size
that changes every step.

This is stage 12. The capture is twenty lines and three of them are traps.

In [ ]:
### run this cell: a stand-in for a decode step

class Layer(nn.Module):
  """One transformer layer, shaped like a decode step: many small kernels."""
  def __init__(self, hidden):
    super().__init__()
    self.attention_norm = nn.RMSNorm(hidden)
    self.qkv_proj = nn.Linear(hidden, 3*hidden, bias=False)
    self.out_proj = nn.Linear(hidden, hidden, bias=False)
    self.mlp_norm = nn.RMSNorm(hidden)
    self.up_proj = nn.Linear(hidden, 4*hidden, bias=False)
    self.down_proj = nn.Linear(4*hidden, hidden, bias=False)

  def forward(self, hidden_states):
    normed = self.attention_norm(hidden_states)
    query, key, value = self.qkv_proj(normed).chunk(3, -1)
    scores = query @ key.transpose(-1, -2) / 32.0
    attention = torch.softmax(scores, -1) @ value
    hidden_states = hidden_states + self.out_proj(attention)
    normed = self.mlp_norm(hidden_states)
    return hidden_states + self.down_proj(F.silu(self.up_proj(normed)))

class Model(nn.Module):
  def __init__(self, num_layers=28, hidden=1024):
    super().__init__()
    self.layers = nn.ModuleList([Layer(hidden) for _ in range(num_layers)])

  def forward(self, hidden_states):
    for layer in self.layers:
      hidden_states = layer(hidden_states)
    return hidden_states

HIDDEN, LAYERS = 1024, 28

def random_input(num_seqs, hidden=HIDDEN):
  """The hidden states of one decode step: one token for each sequence."""
  return torch.randn(num_seqs, 1, hidden, device='cuda', dtype=torch.bfloat16)

model = Model(LAYERS, HIDDEN).cuda().to(torch.bfloat16).eval()
model.requires_grad_(False)
print(f'{LAYERS} layers, hidden size {HIDDEN}')

# Exercise 1: capture and replay

Warm up, record, replay. Measure what it bought.

In [ ]:
def capture(model, example):
  """Record one forward pass as a graph.
  -> (graph, static_input, static_output).

  Warm up on a side stream first. cuBLAS allocates workspaces on the first
  call, and the graph must not record that allocation.
  """
  static_input = example.clone()
  side_stream = torch.cuda.Stream()
  side_stream.wait_stream(torch.cuda.current_stream())
  with torch.cuda.stream(side_stream):
    

  torch.cuda.current_stream().wait_stream(side_stream)
  graph = torch.cuda.CUDAGraph()
  with torch.cuda.graph(graph):
    static_output = 
  return graph, static_input, static_output

with torch.no_grad():
  step_input = random_input(1)
  graph, static_input, static_output = capture(model, step_input)
  eager_ms = 
graph_ms = 
print(f'eager {eager_ms:.3f} ms, graph {graph_ms:.3f} ms  ({eager_ms/graph_ms:.2f}x)')

# Exercise 2: feed it

Replay runs the exact work that the capture recorded. It reads and writes the
exact buffers that the capture recorded. Put your data into those buffers.

In [ ]:
real_input = random_input(1)
# Put real_input into the input of the graph. Caution: the graph holds a POINTER.

graph.replay()
from_graph = static_output.clone()
with torch.no_grad():
  from_eager = model(real_input)
print('agree:', torch.allclose(from_graph, from_eager, rtol=1e-2, atol=1e-2))

# Now do it the wrong way, and look at the result.
static_input = random_input(1)    # a new binding, not a copy
graph.replay()
print('still the OLD output:', torch.allclose(static_output, from_graph))

# Exercise 3: shape buckets

A server's batch size changes every step and a graph's does not. Capture
several and pad up to the nearest.

In [ ]:
BUCKETS = [1, 2, 4, 8, 16, 32]
graphs = {}
with torch.no_grad():
  for bucket in BUCKETS:
    graphs[bucket] = 

def bucket_for(num_seqs):
  """The smallest bucket that holds num_seqs, or None if no bucket does."""
  return 

def run(num_seqs):
  bucket = bucket_for(num_seqs)
  if bucket is None:
    with torch.no_grad():
      return model(random_input(num_seqs))
  graph, static_input, static_output = graphs[bucket]
  # Copy num_seqs rows into the first rows of the bucket input. Replay.
  # Read back only the num_seqs rows that you need.

  return 

print(f"{'seqs':>5} {'bucket':>7} {'padding':>8} {'eager ms':>10} {'graph ms':>10} {'gain':>6}")
for num_seqs in (1, 3, 5, 12, 31, 48):
  step_input = random_input(num_seqs)
  with torch.no_grad():
    eager_ms = cudalib.bench_ms(lambda: model(step_input), iters=30, warmup=10, best_of=2)
  graph_ms = cudalib.bench_ms(lambda: run(num_seqs), iters=30, warmup=10, best_of=2)
  bucket = bucket_for(num_seqs)
  padding = f'{100*(bucket-num_seqs)/bucket:.0f}%' if bucket else 'n/a'
  print(f'{num_seqs:>5} {str(bucket):>7} {padding:>8} {eager_ms:>10.3f} '
        f'{graph_ms:>10.3f} {eager_ms/graph_ms:>5.2f}x')

# Exercise 4: what the buckets cost

Not time. Memory.

In [ ]:
# Each captured graph keeps its buffers for the life of the server.
# Measure the cost of two more buckets.
KV_BYTES_PER_TOKEN = 112 * 1024   # Qwen3-0.6B: 2 x 28 layers x 8 heads x 128 x 2 bytes
free_before = torch.cuda.mem_get_info()[0]
extra_graphs = {}
with torch.no_grad():
  for bucket in (64, 128):
    
free_after = torch.cuda.mem_get_info()[0]
cost_mb = 
print(f'2 more graphs cost {cost_mb:.0f} MB of VRAM')
print(f'at 112 KB of KV cache for each token, the pool loses '
      f'{cost_mb*1e6/KV_BYTES_PER_TOKEN:,.0f} tokens')

### Before you open the solution

1. Remove the warm-up on the side stream in `capture`, and run it again. What
   changes? Does it fail loudly or quietly?
2. The second half of Exercise 2 binds `static_input` to a new tensor and replays. You
   get the old answer, and no error. What exactly does the graph hold?
3. Exercise 4 measured the memory that two more buckets cost. Where does that
   memory come from? What did Part 3 do to the same pool in four sections?
4. A step with 5 sequences runs the bucket for 8. The GPU does three rows of
   arithmetic that nobody uses. Why does that cost almost nothing here? Which
   plot from Part 1 tells you?